# Tune the Product Backorder Model (Kaggle)

This notebook tunes a backorder classifier using the `clu.zip` dataset (`train.csv`, `test.csv`, and optional `sample_Submission.csv`).

**Goal:** improve minority-class detection and select a useful decision threshold using the provided validation split. Compare Logistic Regression and Random Forest with a tuned Random Forest and HistGradientBoosting. Tune with Average Precision, then report precision/recall/F1 at multiple thresholds.

**Important:** this is general product backorder data, not livestock-feed inventory data. Do not present results as validated farm stockout performance. Threshold choice should reflect the cost of false alarms versus missed backorders.

In [ ]:
# 1. Imports
import os, glob, zipfile, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, average_precision_score, precision_recall_curve,
    precision_recall_fscore_support, fbeta_score, make_scorer
)
from sklearn.base import clone
import joblib
from IPython.display import display, FileLink
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
print("Imports ready")

## 2. Load the data

Attach the same dataset to Kaggle with **Add Input**. This cell searches for the CSVs and extracts `clu.zip` if it is present as a file.

In [ ]:
roots = ["/kaggle/input", "/kaggle/working", "."]
train_paths, test_paths, sample_paths, zip_paths = [], [], [], []
for root in roots:
    if os.path.exists(root):
        train_paths += glob.glob(os.path.join(root, "**", "train.csv"), recursive=True)
        test_paths += glob.glob(os.path.join(root, "**", "test.csv"), recursive=True)
        sample_paths += glob.glob(os.path.join(root, "**", "sample_Submission.csv"), recursive=True)
        zip_paths += glob.glob(os.path.join(root, "**", "clu.zip"), recursive=True)

if not train_paths and zip_paths:
    extract_dir = "/kaggle/working/clu_extracted"
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_paths[0]) as z:
        z.extractall(extract_dir)
    train_paths = glob.glob(os.path.join(extract_dir, "**", "train.csv"), recursive=True)
    test_paths = glob.glob(os.path.join(extract_dir, "**", "test.csv"), recursive=True)
    sample_paths = glob.glob(os.path.join(extract_dir, "**", "sample_Submission.csv"), recursive=True)

if not train_paths or not test_paths:
    raise FileNotFoundError("Could not find train.csv and test.csv. Attach clu.zip via Kaggle Add Input.")
train_df = pd.read_csv(train_paths[0])
test_df = pd.read_csv(test_paths[0])
sample_df = pd.read_csv(sample_paths[0]) if sample_paths else None

TARGET = "went_on_backorder"
print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
print("Target distribution:")
display(train_df[TARGET].value_counts())
display(train_df.head())

## 3. Preserve the supplied validation split

Use `Validation == 0` for model fitting and `Validation == 1` for model selection/evaluation. Never include `Validation` or `sku` as predictors. If the split column is missing or has only one value, use a stratified split instead.

In [ ]:
from sklearn.model_selection import train_test_split
if "Validation" in train_df.columns and train_df["Validation"].nunique() >= 2:
    fit_df = train_df[train_df["Validation"] == 0].copy()
    valid_df = train_df[train_df["Validation"] == 1].copy()
    split_note = "Provided Validation flag"
else:
    fit_df, valid_df = train_test_split(
        train_df, test_size=0.25, random_state=RANDOM_STATE,
        stratify=train_df[TARGET]
    )
    split_note = "Stratified random split fallback"

drop_cols = [TARGET, "Validation", "sku"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]
X_fit, y_fit = fit_df[feature_cols], fit_df[TARGET].astype(int)
X_valid, y_valid = valid_df[feature_cols], valid_df[TARGET].astype(int)
print(split_note, "| Fit:", X_fit.shape, "| Validation:", X_valid.shape)
print("Positive rate:", round(y_fit.mean(), 4), "->", round(y_valid.mean(), 4))
print("Features:", feature_cols)

## 4. Build a leakage-safe preprocessing pipeline

These columns are numeric in this dataset. Median imputation is inside the pipeline so preprocessing is learned only from the fit split. Scaling is needed for Logistic Regression, but tree models do not depend on feature scaling.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_cols)
])

# Baseline reference
baseline = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", DummyClassifier(strategy="prior"))
])
baseline.fit(X_fit, y_fit)
baseline_proba = baseline.predict_proba(X_valid)[:, 1]
print("Baseline Average Precision:", round(average_precision_score(y_valid, baseline_proba), 4))

## 5. Tune Random Forest with randomized cross-validation

Search uses Average Precision (a useful metric for imbalanced binary classification). This can take a few minutes depending on Kaggle CPU availability. `n_iter` is deliberately moderate; increase it if you have more compute time.

In [ ]:
rf_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 8, 14, 22],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", 0.5, 0.8],
    "model__class_weight": ["balanced", "balanced_subsample"]
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=param_distributions,
    n_iter=18,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True
)
start = time.time()
search.fit(X_fit, y_fit)
print("Tuning seconds:", round(time.time()-start, 1))
print("Best CV Average Precision:", round(search.best_score_, 4))
print("Best parameters:")
print(search.best_params_)
tuned_rf = search.best_estimator_

## 6. Compare tuned model with Logistic Regression and Extra Trees

These comparisons use the same fit/validation split. The validation set should not be used repeatedly to make unlimited modeling choices; treat this as model selection and report the held-out metrics transparently.

In [ ]:
lr_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols)
    ])),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
])
extra_pipe = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=2,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ))
])
lr_pipe.fit(X_fit, y_fit)
extra_pipe.fit(X_fit, y_fit)

candidate_models = {
    "Logistic Regression": lr_pipe,
    "Previous-style Extra Trees": extra_pipe,
    "Tuned Random Forest": tuned_rf
}
comparison_rows = []
probabilities = {}
for name, model in candidate_models.items():
    prob = model.predict_proba(X_valid)[:, 1]
    probabilities[name] = prob
    pred = (prob >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_valid, pred, average="binary", zero_division=0)
    comparison_rows.append({
        "model": name,
        "precision@0.5": p,
        "recall@0.5": r,
        "f1@0.5": f1,
        "roc_auc": roc_auc_score(y_valid, prob),
        "average_precision": average_precision_score(y_valid, prob)
    })
comparison = pd.DataFrame(comparison_rows).sort_values("average_precision", ascending=False)
display(comparison)
best_name = comparison.iloc[0]["model"]
best_model = candidate_models[best_name]
best_prob = probabilities[best_name]
print("Selected by validation average precision:", best_name)

## 7. Choose a decision threshold

A model probability is not automatically a yes/no decision. The default threshold 0.50 may miss too many backorders. The table below shows precision, recall, F1 and F2 at several thresholds. F2 gives recall more weight than precision, useful when missing a backorder is considered more costly than investigating a false alarm. Choose the threshold that matches the project's operational priorities; do not select it on the test set.

In [ ]:
threshold_rows = []
for threshold in np.round(np.arange(0.10, 0.91, 0.05), 2):
    pred = (best_prob >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_valid, pred, average="binary", zero_division=0)
    threshold_rows.append({
        "threshold": threshold,
        "precision": p,
        "recall": r,
        "f1": f1,
        "f2_recall_weighted": fbeta_score(y_valid, pred, beta=2, zero_division=0),
        "flagged_fraction": pred.mean()
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
threshold_table.set_index("threshold")[["precision", "recall", "f1", "f2_recall_weighted"]].plot(
    marker="o", figsize=(10,5), title=f"Threshold trade-offs — {best_name}"
)
plt.ylabel("Score")
plt.grid(True)
plt.show()

# Example threshold: choose the one maximizing F2 on validation.
# For production, choose based on the real cost of missed backorders vs false alarms.
best_threshold_row = threshold_table.loc[threshold_table["f2_recall_weighted"].idxmax()]
chosen_threshold = float(best_threshold_row["threshold"])
print("F2-selected threshold:", chosen_threshold)
print(best_threshold_row.to_dict())

In [ ]:
# Final report at the selected threshold
chosen_pred = (best_prob >= chosen_threshold).astype(int)
print("Selected threshold:", chosen_threshold)
print(classification_report(
    y_valid, chosen_pred, target_names=["No backorder", "Backorder"],
    digits=4, zero_division=0
))
ConfusionMatrixDisplay.from_predictions(
    y_valid, chosen_pred, display_labels=["No backorder", "Backorder"]
)
plt.title(f"{best_name} at threshold {chosen_threshold:.2f}")
plt.show()
print("ROC-AUC:", round(roc_auc_score(y_valid, best_prob), 4))
print("Average Precision:", round(average_precision_score(y_valid, best_prob), 4))

## 8. Fit the selected model on all labelled data and predict test rows

The validation metrics above are the ones to report. The final artifact is refit on all labelled rows for later inference/submission. Its training fit is not a new independent evaluation.

In [ ]:
X_all = train_df[feature_cols]
y_all = train_df[TARGET].astype(int)
final_model = clone(best_model)
final_model.fit(X_all, y_all)

test_features = test_df.reindex(columns=feature_cols)
test_prob = final_model.predict_proba(test_features)[:, 1]
test_class = (test_prob >= chosen_threshold).astype(int)

test_results = test_df.copy()
test_results["predicted_backorder_probability"] = test_prob
test_results["predicted_went_on_backorder"] = test_class
display(test_results.head())

## 9. Save artifacts

In [ ]:
out_dir = "/kaggle/working/tuned_backorder_model_outputs"
os.makedirs(out_dir, exist_ok=True)
joblib.dump(final_model, os.path.join(out_dir, "tuned_backorder_risk_model.joblib"))
comparison.to_csv(os.path.join(out_dir, "model_comparison.csv"), index=False)
threshold_table.to_csv(os.path.join(out_dir, "threshold_analysis.csv"), index=False)
test_results.to_csv(os.path.join(out_dir, "test_predictions.csv"), index=False)

validation_summary = {
    "dataset": "Product backorder dataset (clu.zip)",
    "split": split_note,
    "selected_model": best_name,
    "chosen_threshold": chosen_threshold,
    "validation_roc_auc": float(roc_auc_score(y_valid, best_prob)),
    "validation_average_precision": float(average_precision_score(y_valid, best_prob)),
    "validation_precision_at_threshold": float(precision_recall_fscore_support(y_valid, chosen_pred, average="binary", zero_division=0)[0]),
    "validation_recall_at_threshold": float(precision_recall_fscore_support(y_valid, chosen_pred, average="binary", zero_division=0)[1]),
    "validation_f1_at_threshold": float(precision_recall_fscore_support(y_valid, chosen_pred, average="binary", zero_division=0)[2]),
    "best_random_forest_cv_average_precision": float(search.best_score_),
    "best_random_forest_parameters": search.best_params_,
    "warning": "General product backorder model, not validated for livestock-feed stockouts."
}
with open(os.path.join(out_dir, "validation_summary.json"), "w") as f:
    json.dump(validation_summary, f, indent=2)

zip_path = "/kaggle/working/tuned_backorder_prediction_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(out_dir):
        z.write(os.path.join(out_dir, fn), arcname=fn)
print("Saved:", zip_path)
display(FileLink(zip_path))

## 10. Interpretation

- Compare average precision and recall, not accuracy alone, because backorders are the minority class.
- A lower threshold usually catches more backorders but creates more false alarms.
- Threshold selection should reflect how costly missed backorders are relative to unnecessary procurement reviews.
- This is real general supply-chain data, not livestock-feed inventory. Use it as a prototype only until farm-specific records are available.